# End-to-End Pipeline — New Config Format

Verifies the new `[Observation]`/`[Fastcam]`/`[Slowcam]`/`[ROI]`/`[Outputs]` config schema.

Tutorial data: PSF is the faster camera (`Fastcam.type = PSF`).

Dark routing:
- `Fastcam.dark` (PSF dark) → subtracted in **sort** step
- `Slowcam.dark` (PL dark)  → subtracted in **ingest** step

`apply_deadtime_correction` is always True (not in config).

In [ ]:
import sys, os
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import h5py
from astropy.io import fits
from configobj import ConfigObj

PLRED_ROOT = Path('__file__').resolve().parent.parent
if str(PLRED_ROOT) not in sys.path:
    sys.path.insert(0, str(PLRED_ROOT))

from PLred.sort import script_match_timestamps
from PLred.ingest import ingest_from_config_unified
from PLred.average import build_ROI_access_from_config, average_to_h5_from_config, explore_grid
import PLred.specextract as specextract
from PLred.specextract import extract_from_config
from PLred.scripts._diagnostics import plot_first_frame_check

TUTORIALS_DIR = Path('/Users/yjkim/Documents/PLred2/PLred-dev/PLred/tutorials')
OUTPUT_DIR    = TUTORIALS_DIR / 'tutorial_output_newcfg'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

OBS_CONFIG = OUTPUT_DIR / 'obs_new.ini'

FASTCAM_DIR = TUTORIALS_DIR / 'data' / 'fastcam'
SLOWCAM_DIR = TUTORIALS_DIR / 'data' / 'slowcam'
FLAT_FITS   = SLOWCAM_DIR / 'cropped_firstpl_15:05:17.315924371.fits'

print('OUTPUT_DIR  :', OUTPUT_DIR)
print('OBS_CONFIG  :', OBS_CONFIG)

## Write the new-format config

Key differences from the old format:
- `[Fastcam]` / `[Slowcam]` carry `type`, `timestamp_dir`, `data_dir`, `dark`
- `[ROI]` carries `PLcam_ROI` and `PSFcam_crop_width`
- `[Outputs]` has all output file names
- No `[Sort]`, `[Ingest]`, `[Instrument]`, `[Cameras]` sections
- `apply_deadtime_correction` not present → always True

In [ ]:
cfg = ConfigObj()
cfg.filename = str(OBS_CONFIG)

cfg['Observation'] = {
    'obsdate'   : '20240917',
    'start_time': '15:05:10',
    'end_time'  : '15:05:11',
}

cfg['Fastcam'] = {
    'type'         : 'PSF',
    'timestamp_dir': str(FASTCAM_DIR) + '/',
    'data_dir'     : str(FASTCAM_DIR) + '/',
    'dark'         : str(FASTCAM_DIR / 'dark.fits'),
}

cfg['Slowcam'] = {
    'type'         : 'PL',
    'timestamp_dir': str(SLOWCAM_DIR) + '/',
    'data_dir'     : str(SLOWCAM_DIR) + '/',
    'dark'         : str(SLOWCAM_DIR / 'dark.fits'),
    'slowcam_nbin' : '1',
}

cfg['ROI'] = {
    'PLcam_ROI'       : '0,412,1200,1220',
    'PSFcam_crop_width': '20',
}

cfg['Spectrum'] = {
    'nfib'               : '38',
    'spectral_orientation': 'horizontal',
}

cfg['ROIViewer'] = {
    'roi': '0,412,1200,1220',
}

cfg['Average'] = {
    'map_n'      : '5',
    'map_width'  : '2',
    'xc'         : '18.26',
    'yc'         : '20.50',
    'pix2mas'    : '16.2',
    'n_bootstrap': '0',
}

cfg['Specextract'] = {
    'extractor' : 'trace_box',
    'trace_file': str(OUTPUT_DIR / 'traces.npz'),
    'truncate'  : '0',
}

cfg['Outputs'] = {
    'timestamp_match_output': str(OUTPUT_DIR / 'fastcam.h5'),
    'ingest_output'         : str(OUTPUT_DIR / 'alldata.h5'),
    'ROI_viewer_output'     : str(OUTPUT_DIR / 'roi.h5'),
    'average_output'        : str(OUTPUT_DIR / 'map.h5'),
    'couplingmap_output'    : str(OUTPUT_DIR / 'couplingmap.fits'),
}

cfg.write()
print('Wrote config to:', OBS_CONFIG)
print()
with open(OBS_CONFIG) as fh:
    print(fh.read())

## Step 1: Timestamp matching

PSF dark (`Fastcam.dark`) subtracted from PSF frames here.  
Dead-time correction always applied.

In [ ]:
%%time
print('Running sort (timestamp matching) ...')
script_match_timestamps(str(OBS_CONFIG))

h5_sort = str(OUTPUT_DIR / 'fastcam.h5')
with h5py.File(h5_sort, 'r') as f:
    frames     = f['frames'][:]
    nstacks    = f['nstacks'][:]
    timestamps = f['timestamps'][:]
    import json
    meta = json.loads(f['metadata'][()])

print(f'\nfastcam.h5: {os.path.getsize(h5_sort)/1e6:.1f} MB')
print(f'  n_frames        : {len(frames)}')
print(f'  frame shape     : {frames.shape[1:]}')
print(f'  psfcam_is_fast  : {meta["psfcam_is_fast"]}')
print(f'  dead_time_corr  : {meta["dead_time_corrected"]}')
print(f'  nstacks mean    : {nstacks.mean():.2f}')

assert meta['psfcam_is_fast'] == True,  'psfcam_is_fast should be True (Fastcam.type=PSF)'
assert meta['dead_time_corrected'] == True, 'dead-time correction should always be applied'
print('\n✓ Assertions passed')

In [ ]:
# PSF frame mosaic
n = len(frames)
idxs = np.linspace(0, n-1, min(6, n), dtype=int)
fig, axes = plt.subplots(2, 3, figsize=(10, 6))
for ax, idx in zip(axes.flat, idxs):
    im = ax.imshow(frames[idx], origin='lower', cmap='inferno', vmin=0)
    ax.set_title(f'PSF frame {idx}')
    ax.axis('off')
    plt.colorbar(im, ax=ax, fraction=0.046)
fig.suptitle('Step 1: PSFcam weighted-mean frames (dark-subtracted)')
plt.tight_layout(); plt.show()

print('Background (corner 5×5) mean:', frames[0, :5, :5].mean())
print('PSF peak:', frames[0].max())

## Step 2: Ingest

PL dark (`Slowcam.dark`) subtracted from PL frames here (because `Fastcam.type = PSF`).  
First-frame check shows both PSF and PL frames with `vmin=0` to verify dark subtraction.

In [ ]:
%%time
print('Running ingest ...')
ingest_from_config_unified(str(OBS_CONFIG))

alldata_h5 = str(OUTPUT_DIR / 'alldata.h5')
print(f'\nalldata.h5: {os.path.getsize(alldata_h5)/1e6:.1f} MB')

with h5py.File(alldata_h5, 'r') as f:
    print('plcam/frames shape    :', f['plcam/frames'].shape)
    print('psfcam/frames shape   :', f['psfcam/frames'].shape)
    print('plcam dark_subtracted :', f['plcam'].attrs.get('dark_subtracted', '?'))
    print('plcam dark_source     :', f['plcam'].attrs.get('dark_source', '?'))
    print('plcam_roi             :', list(f.attrs.get('plcam_roi', [])))

assert f['plcam'].attrs.get('dark_subtracted') == True, 'PL dark should be applied in ingest'
print('\n✓ PL dark applied in ingest')

In [ ]:
# First-frame diagnostic: PSF (cropped) and PL ROI side-by-side
# vmin=0 confirms dark subtraction; background should be near zero
fig = plot_first_frame_check(alldata_h5, pix2mas=16.2)
fig.savefig(str(OUTPUT_DIR / '2_first_frame.png'), dpi=120, bbox_inches='tight')
plt.show(fig)
print('Saved to:', OUTPUT_DIR / '2_first_frame.png')

## Step 3: ROI viewer cache

In [ ]:
%%time
print('Building ROI access cache ...')
build_ROI_access_from_config(str(OBS_CONFIG))

roi_h5 = str(OUTPUT_DIR / 'roi.h5')
print(f'roi.h5: {os.path.getsize(roi_h5)/1e6:.1f} MB')

## Step 4: Spatial averaging

In [ ]:
%%time
print('Running spatial averaging ...')
average_to_h5_from_config(str(OBS_CONFIG))

map_h5 = str(OUTPUT_DIR / 'map.h5')
print(f'map.h5: {os.path.getsize(map_h5)/1e6:.2f} MB')

with h5py.File(map_h5, 'r') as f:
    print('avg_PLcam shape:', f['avg_PLcam'].shape)
    print('nframes:\n', f['metadata/nframes'][:])

## Step 5: Trace extraction + spectral extraction

In [ ]:
traces_out = str(OUTPUT_DIR / 'traces.npz')
specextract.make_trace_file(
    fits_path   = str(FLAT_FITS),
    nfib        = 38,
    dark_path   = str(SLOWCAM_DIR / 'dark.fits'),
    xmin        = 1200,
    xmax        = 1220,
    thres       = 0.05,
    min_dist    = 6,
    trace_width = 4,
    poly_deg    = 2,
    outpath     = traces_out,
    plot        = False,
    verbose     = True,
)
print('traces.npz written')

In [ ]:
%%time
print('Running spectral extraction ...')
extract_from_config(str(OBS_CONFIG))

cm_fits = str(OUTPUT_DIR / 'couplingmap.fits')
print(f'couplingmap.fits: {os.path.getsize(cm_fits)/1e6:.2f} MB')

with fits.open(cm_fits) as hdl:
    hdl.info()
    cm = hdl[0].data
    print('Coupling map shape:', cm.shape)

In [ ]:
# Quick sanity: coupling map should have non-zero entries for populated bins
with h5py.File(map_h5, 'r') as f:
    nframes = f['metadata/nframes'][:]

populated = nframes > 0
print(f'Populated bins: {populated.sum()} / {populated.size}')
print(f'Coupling map min (populated): {cm[populated].min():.3f}')
print(f'Coupling map max (populated): {cm[populated].max():.3f}')

assert populated.sum() > 0, 'No populated bins found'
assert cm[populated].max() > 0, 'Coupling map is all zeros — extraction failed'
print('\n✓ Full pipeline verification passed with new config format')